# Tokenisers

In [2]:
from huggingface_hub import login
from transformers import AutoTokenizer
from dotenv import load_dotenv
import os

In [3]:
load_dotenv(override=True)

open_ai_key=os.getenv("OPENAI_API_KEY")
hf_token=os.getenv("HF_TOKEN")

login(hf_token,add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


What does this AutoTokeniser actually mean ->
 A tokenizer is a program that converts human-readable text into tokens (numbers) that an LLM understands.


You type:
"Hello, how are you?"

        ↓ Tokenizer

["Hello", ",", "how", "are", "you", "?"]

        ↓ Converts to IDs

[9906, 11, 1268, 527, 499, 30]

        ↓

LLM processes numbers, not words.

AutoTokenizer

        │
        ▼

Looks at model name

        │
        ▼

"meta-llama/Meta-Llama-3.1-8B"

        │
        ▼

"Oh...
This model needs LlamaTokenizer"

        │
        ▼

Downloads tokenizer.json
special_tokens_map.json
tokenizer_config.json
vocab

        │
        ▼

Returns a ready-to-use tokenizer



Definition:
AutoTokenizer is a Hugging Face factory class that automatically loads the correct tokenizer for a given pretrained model.

Purpose:
Converts human-readable text into token IDs (and back) using the tokenizer that the model was trained with.

Why "Auto"?
You don't need to know whether the model uses LlamaTokenizer, GPT2Tokenizer, BertTokenizer, etc. AutoTokenizer detects and loads the correct one automatically.

⚠️ Common Confusion

AutoTokenizer does not perform tokenization itself.

It is responsible for loading the appropriate tokenizer class. The loaded tokenizer then performs the actual encoding and decoding.

In [4]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)

In [5]:
text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens

[128000,
 40,
 1097,
 12304,
 311,
 1501,
 9857,
 12509,
 304,
 1957,
 311,
 856,
 445,
 11237,
 25175]

In [6]:
character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f"There are {character_count} characters, {word_count} words and {token_count} tokens")

There are 61 characters, 12 words and 15 tokens


In [7]:
tokenizer.decode(tokens)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'<|begin_of_text|>I am excited to show Tokenizers in action to my LLM engineers'

tokenizer.batch_decode()

Definition:
Converts multiple sequences of token IDs back into human-readable text.

Purpose:
Decode a batch of model outputs at once

Key Takeaway:

decode() → one sequence.

batch_decode() → multiple sequences.

In [8]:
tokenizer.batch_decode(tokens)

['<|begin_of_text|>I am excited to show Tokenizers in action to my LLM engineers']

tokenizer.vocab

Definition:
A dictionary containing the tokenizer's base vocabulary (Token → Token ID).

Purpose:
View all tokens that the tokenizer originally knows.

Example:

tokenizer.vocab["hello"]

Output:

9906

Key Takeaway:

vocab = Original vocabulary of the pretrained tokenizer.

In [9]:
# tokenizer.vocab
tokenizer.get_added_vocab()

{'<|begin_of_text|>': 128000,
 '<|end_of_text|>': 128001,
 '<|reserved_special_token_0|>': 128002,
 '<|reserved_special_token_1|>': 128003,
 '<|finetune_right_pad_id|>': 128004,
 '<|reserved_special_token_2|>': 128005,
 '<|start_header_id|>': 128006,
 '<|end_header_id|>': 128007,
 '<|eom_id|>': 128008,
 '<|eot_id|>': 128009,
 '<|python_tag|>': 128010,
 '<|reserved_special_token_3|>': 128011,
 '<|reserved_special_token_4|>': 128012,
 '<|reserved_special_token_5|>': 128013,
 '<|reserved_special_token_6|>': 128014,
 '<|reserved_special_token_7|>': 128015,
 '<|reserved_special_token_8|>': 128016,
 '<|reserved_special_token_9|>': 128017,
 '<|reserved_special_token_10|>': 128018,
 '<|reserved_special_token_11|>': 128019,
 '<|reserved_special_token_12|>': 128020,
 '<|reserved_special_token_13|>': 128021,
 '<|reserved_special_token_14|>': 128022,
 '<|reserved_special_token_15|>': 128023,
 '<|reserved_special_token_16|>': 128024,
 '<|reserved_special_token_17|>': 128025,
 '<|reserved_special_to

# Here we can Clearly see some demonstrations that map to the reserved Token Ids for some tasks that instruct the model like for eg <start_of_the_prompt> has token id = 128000

In [10]:
len(tokenizer.vocab)

128256

# Instruct variants of models

Many models have a variant that has been trained for use in Chats.  
These are typically labelled with the word "Instruct" at the end.  
They have been trained to expect prompts with a particular format that includes system, user and assistant prompts.  

There is a utility method `apply_chat_template` that will convert from the messages list format we are familiar with, into the right input prompt for this model.

In [11]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## Crucial "Aha" moment

For 2.5 weeks, I've given you the impression that LLMs could receive a list of python dictionaries in some way:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
```

But an LLM is just a Data Science model that takes a sequence of numbers and predicts the probability of the next number! You can't pass a bunch of Python objects into a statistical model!

### And now you have the missing piece of the puzzle..

The messages in OpenAI format get converted:

1. ...into a sequence of words with special tags to separate the System, User, Assistant prompt
2. ...then the words are broken down into fragments - "tokens"
3. ...then the tokens are replaced with Token IDs - and this is the input sequence

> The input to an LLM is a sequence of Token IDs. The output is the probability distribution of the next Token ID to follow this input.

That's it!


# Trying new models

We will now work with 3 models:

Phi4 from Microsoft  
DeepSeek 3.1 from DeepSeek AI  
QwenCoder 2.5 from Alibaba Cloud

In [14]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [18]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)
text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print("Llama:")
tokens = tokenizer.encode(text)
print(tokens)
print(tokenizer.batch_decode(tokens))
print("\nPhi 4:")
tokens = phi4_tokenizer.encode(text)
print(tokens)
print(phi4_tokenizer.batch_decode(tokens))

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Llama:
[128000, 40, 1097, 2917, 13610, 12304, 311, 1501, 473, 36368, 19109, 9857, 12509, 304, 1957, 311, 856, 445, 11237, 25175]
['<|begin_of_text|>I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers']

Phi 4:
[40, 939, 4396, 23138, 15209, 316, 2356, 59116, 4512, 29049, 17951, 24223, 306, 3736, 316, 922, 451, 19641, 32437]
['I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers']


In [19]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Phi 4:
<|system|>You are a helpful assistant<|end|><|user|>Tell a light-hearted joke for a room of Data Scientists<|end|><|assistant|>


In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print(tokenizer.encode(text))
print()
print(phi4_tokenizer.encode(text))
print()
print(deepseek_tokenizer.encode(text))

In [16]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Phi:


NameError: name 'phi4_tokenizer' is not defined

In [17]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code)
for token in tokens:
    print(f"{token}={qwen_tokenizer.decode(token)}")

198=

750=def
23811= hello
31792=_world
29766=(person
982=):

220= 
1173= print
445=("
9707=Hello
497=",
1697= person
340=)



# When using hosted APIs (OpenAI, Anthropic, OpenRouter), the provider formats your messages into the model's expected prompt internally. When running an Instruct model locally with Hugging Face, there is no provider layer, so you must call apply_chat_template() yourself before tokenization.

# This is why you'll rarely see apply_chat_template() in API examples, but you'll almost always see it in Hugging Face examples that run models locally.

You're sending it to the OpenAI/OpenRouter server.

The server internally does something like:

messages
      ↓
Apply Chat Template
      ↓
Tokenize
      ↓
Model

This preprocessing is hidden from you.



# You're sending it to the OpenAI/OpenRouter server.

The server internally does something like:

messages
      ↓
Apply Chat Template
      ↓
Tokenize
      ↓
Model

This preprocessing is hidden from you.

tokenizer.apply_chat_template(messages)

does.